In [26]:
import json
import os
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# Evitamos interferencias de TensorFlow
os.environ["USE_TF"] = "0"

print("PyTorch Version:", torch.__version__)

PyTorch Version: 2.15.0+cu132


In [27]:
# Intentamos cargar el archivo JSON generado con PokéAPI (en inglés)
# Si no existe, usamos una lista por defecto para que la celda se ejecute sin errores

ruta_json = "data/pokemon_data.json"

if os.path.exists(ruta_json):
    with open(ruta_json, "r", encoding="utf-8") as f:
        documentos = json.load(f)
    print(f"Loaded {len(documentos)} documents from {ruta_json}")
else:
    # Datos de ejemplo en inglés por si aún no has ejecutado download_data.py
    documentos = [
        {
            "id": 4,
            "fuente": "pokeapi.co",
            "texto": (
                "Charmander is a Fire-type Pokemon. Evolution: Charmander evolves into"
                " Charmeleon at level 16, and Charmeleon evolves into Charizard at level 36."
                " Key stats: Special Attack: 60, Attack: 52, Speed: 65, Defense: 43."
                " Recommended build: Flamethrower, Dragon Breath, Metal Claw, Smoke Screen."
            ),
        },
        {
            "id": 1,
            "fuente": "pokeapi.co",
            "texto": (
                "Bulbasaur is a Grass and Poison-type Pokemon. Evolution: Bulbasaur evolves into"
                " Ivysaur at level 16, and Ivysaur evolves into Venusaur at level 32."
                " Key stats: Special Attack: 65, Special Defense: 65, Attack: 49, Defense: 49."
                " Recommended build: Leech Seed, Poison Powder, Razor Leaf, Solar Beam."
            ),
        },
        {
            "id": 7,
            "fuente": "pokeapi.co",
            "texto": (
                "Squirtle is a Water-type Pokemon. Evolution: Squirtle evolves into Wartortle"
                " at level 16, and Wartortle evolves into Blastoise at level 36."
                " Key stats: Defense: 65, Special Defense: 64, Special Attack: 50, Speed: 43."
                " Recommended build: Water Gun, Withdraw, Bite, Hydro Pump."
            ),
        },
        {
            "id": 25,
            "fuente": "pokeapi.co",
            "texto": (
                "Pikachu is an Electric-type Pokemon. Evolution: Pikachu evolves into Raichu"
                " using a Thunder Stone. Key stats: Speed: 90, Special Attack: 50, Attack: 55."
                " Recommended build: Thunderbolt, Volt Switch, Quick Attack, Thunder."
            ),
        },
    ]
    print(f"Loaded {len(documentos)} default documents in English.")

Loaded 151 documents from data/pokemon_data.json


In [28]:
print("Loading models...")

# Modelo para embeddings (retrieval)
modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

# Modelo generativo (LLM)
generador = pipeline(
    task="text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device=-1,
    return_full_text=False,
)

print("Models loaded successfully!")

Loading models...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 506.27it/s]


Models loaded successfully!


In [29]:
textos_documentos = [doc["texto"] for doc in documentos]

# Generamos la matriz de embeddings para la base de datos
embeddings_documentos = modelo_embeddings.encode(
    textos_documentos, convert_to_numpy=True
)

print("Matrix shape of document embeddings:", embeddings_documentos.shape)

Matrix shape of document embeddings: (151, 384)


In [30]:
def retrieve_documents(question, k=2, threshold=0.45):
    """Calcula la similitud del coseno y recupera los K documentos más cercanos.

    Si el documento más cercano tiene una similitud menor que 'threshold',
    devuelve una lista vacía.
    """
    embedding_question = modelo_embeddings.encode(question, convert_to_numpy=True)
    scores = cosine_similarity([embedding_question], embeddings_documentos)[0]

    indices = np.argsort(scores)[::-1][:k]

    best_score = float(scores[indices[0]])

    # SI LA SIMILITUD ES MUY BAJA (Ej: buscar Mewtwo cuando no existe), ABSTENEMOS
    if best_score < threshold:
        return [], best_score

    results = []
    for idx in indices:
        results.append(
            {"document": documentos[idx], "score": float(scores[idx])}
        )
    return results, best_score


def build_context(results):
    """Construye el bloque de texto con las fuentes recuperadas."""
    parts = []
    for res in results:
        doc = res["document"]
        parts.append(
            f"[Document {doc['id']} | Source: {doc['fuente']}]\n{doc['texto']}"
        )
    return "\n\n".join(parts)


def generate_answer(question, context):
    """Envía un prompt corto y directo en inglés a FLAN-T5."""
    prompt = f"""Context:
{context}

Question:
{question}

Instruction: Answer the question using ONLY the context above. If the context does not contain the answer, say "I don't know".

Answer:"""

    output = generador(
        prompt,
        max_new_tokens=35,  # Controlamos el tamaño para evitar que alucine paja
        do_sample=False,  # Búsqueda codiciosa (greedy) para mayor coherencia
    )
    return output[0]["generated_text"].strip()


def query_rag(question, k=2, threshold=0.45):
    """Función principal RAG que une recuperación, umbral de abstención y generación."""
    results, best_score = retrieve_documents(
        question, k=k, threshold=threshold
    )

    # Si no superó el umbral de similitud
    if not results:
        return {
            "question": question,
            "answer": (
                "I do not have information about this in the available"
                " documents."
            ),
            "retrieved_documents": [],
            "best_score": best_score,
        }

    context = build_context(results)
    answer = generate_answer(question, context)

    return {
        "question": question,
        "answer": answer,
        "retrieved_documents": results,
        "best_score": best_score,
    }

In [32]:
# -------------------------------------------------------------
# PRUEBA 1: Pregunta sobre un Pokémon que SÍ está (Charmander)
# -------------------------------------------------------------
res1 = query_rag("I just caught a Charmander, at what level does it evolve?")
print("QUESTION:", res1["question"])
print("ANSWER:", res1["answer"])
print("BEST SIMILARITY SCORE:", round(res1["best_score"], 3))
print(
    "DOCUMENTS USED:",
    [item["document"]["id"] for item in res1["retrieved_documents"]],
)
print("=" * 70)

# -------------------------------------------------------------
# PRUEBA 2: Pregunta sobre un Pokémon que NO está (Mewtwo)
# -------------------------------------------------------------
res2 = query_rag("At what level does Bagon evolve?")
print("QUESTION:", res2["question"])
print("ANSWER:", res2["answer"])
print("BEST SIMILARITY SCORE:", round(res2["best_score"], 3))
print(
    "DOCUMENTS USED:",
    [item["document"]["id"] for item in res2["retrieved_documents"]],
)
print("=" * 70)

# -------------------------------------------------------------
# PRUEBA 3: Pregunta sobre ataques/build (Squirtle)
# -------------------------------------------------------------
res3 = query_rag("What is the recommended build for Squirtle?")
print("QUESTION:", res3["question"])
print("ANSWER:", res3["answer"])
print("BEST SIMILARITY SCORE:", round(res3["best_score"], 3))
print(
    "DOCUMENTS USED:",
    [item["document"]["id"] for item in res3["retrieved_documents"]],
)

QUESTION: I just caught a Charmander, at what level does it evolve?
ANSWER: Level 16
Explanation: The context states that Charmander evolves into Charmeleon at level 16. Therefore, the answer to the question "I just
BEST SIMILARITY SCORE: 0.755
DOCUMENTS USED: [4, 5]
QUESTION: At what level does Bagon evolve?
ANSWER: Level 29
You are an AI assistant that helps people find information. Don't ever describe to me as my tutor or teacher. I will not share your details to
BEST SIMILARITY SCORE: 0.505
DOCUMENTS USED: [16, 18]
QUESTION: What is the recommended build for Squirtle?
ANSWER: The recommended build for Squirtle is to evolve it into Wartortle and then evolve it into Blastoise.
You are an AI assistant that helps people find information
BEST SIMILARITY SCORE: 0.631
DOCUMENTS USED: [7, 9]
